# ANN-SNN Scaling Laws: Full Experiment Sweep (Colab GPU)

This notebook runs the complete 600-experiment sweep on a free T4 GPU.

**Runtime:** ~2-4 hours on T4 GPU

**Steps:**
1. Clone repo & install dependencies
2. Train all ANN checkpoints
3. Run full conversion + evaluation sweep
4. Analyze results & fit scaling laws
5. Generate plots
6. Download results

## 0. Verify GPU

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected! Go to Runtime > Change runtime type > T4 GPU")

## 1. Clone Repo & Install Dependencies

In [ ]:
!git clone https://github.com/Vighneshwarkuru/ann_snn_scaling_laws.git
%cd ann_snn_scaling_laws

In [ ]:
!pip install -q torch torchvision spikingjelly scipy numpy pandas matplotlib seaborn pyyaml ptflops tqdm psutil

In [ ]:
# Verify all imports work
import sys
sys.path.insert(0, '.')
from src.models import get_model
from src.data.datasets import get_dataloaders, get_dataset_info
from src.training.train_ann import load_or_train_ann
from src.conversion.ann_to_snn import ANNtoSNNConverter
from src.evaluation.evaluate_ann import evaluate_ann
from src.evaluation.evaluate_snn import evaluate_snn
from spikingjelly.activation_based import ann2snn
print("All imports OK!")

## 2. Remove Synthetic Results

Delete the demo/synthetic data so we start fresh with real experiments.

In [ ]:
import shutil, os

# Remove synthetic results
for folder in ['results/raw', 'results/aggregated', 'results/scaling_laws', 'results/plots']:
    if os.path.exists(folder):
        shutil.rmtree(folder)
        print(f"Removed: {folder}")

# Remove old checkpoints (we'll train fresh)
if os.path.exists('checkpoints'):
    shutil.rmtree('checkpoints')
    print("Removed: checkpoints")

os.makedirs('results/raw', exist_ok=True)
os.makedirs('checkpoints', exist_ok=True)
print("\nReady for real experiments!")

## 3. Train All ANN Checkpoints

This trains 5 models × 5 datasets = 25 ANNs. Takes ~30-60 min on T4.

In [ ]:
!python scripts/train_all_anns.py --config configs/experiment.yaml

## 4. Run Full Experiment Sweep

5 models × 5 datasets × 8 timesteps × 3 seeds = 600 experiments.

Takes ~1-3 hours on T4. Results are saved incrementally, so you can interrupt and resume.

In [ ]:
!python scripts/run_sweep.py --config configs/experiment.yaml

In [ ]:
# Check how many results we have
import glob
csvs = glob.glob('results/raw/*.csv')
print(f"Completed experiments: {len(csvs)} / 600")

## 5. Analyze Results & Fit Scaling Laws

In [ ]:
!python scripts/analyze_results.py --results-dir results

## 6. Generate All Plots

In [ ]:
!python scripts/generate_plots.py --results-dir results

In [ ]:
# Also generate comparison plots
!python scripts/generate_comparison_plots.py

## 7. View Key Results

In [ ]:
import pandas as pd
import json

# Load scaling law fits
with open('results/scaling_laws/scaling_law_fits.json') as f:
    laws = json.load(f)

# CSI Power Law
if 'CSI_power_law' in laws:
    csi = laws['CSI_power_law']
    print("="*60)
    print("CSI POWER LAW (Main Result)")
    print("="*60)
    print(f"Equation: {csi['equation']}")
    print(f"R² = {csi['r_squared']:.4f}")
    print(f"β (timestep exponent) = {csi['params']['beta']:.3f}")
    print(f"γ (depth exponent) = {csi['params']['gamma']:.3f}")
    print(f"δ (dataset complexity exponent) = {csi['params']['delta']:.3f}")

print("\n")

# Load aggregated results
df = pd.read_csv('results/aggregated/per_run_enriched.csv')
print(f"Total experiments: {len(df)}")
print(f"Models: {sorted(df['model'].unique())}")
print(f"Datasets: {sorted(df['dataset'].unique())}")
print(f"Timesteps: {sorted(df['T'].unique())}")

# Best results per dataset
print("\n" + "="*60)
print("BEST SNN ACCURACY PER DATASET")
print("="*60)
for ds in sorted(df['dataset'].unique()):
    sub = df[df['dataset'] == ds]
    best = sub.loc[sub['snn_accuracy'].idxmax()]
    print(f"{ds:20s}: {best['snn_accuracy']*100:.2f}% ({best['model']}, T={int(best['T'])})")

In [ ]:
# Display plots inline
from IPython.display import Image, display
import os

plot_dir = 'results/plots'
key_plots = [
    'Fig1_CSI_loglog_curves.png',
    'Fig5_accuracy_saturation.png',
    'Fig2_SCI_heatmap.png',
    'Fig3_pareto_3D.png',
]

for p in key_plots:
    path = os.path.join(plot_dir, p)
    if os.path.exists(path):
        print(f"\n{'='*40}")
        print(f"{p}")
        print(f"{'='*40}")
        display(Image(filename=path, width=700))

## 8. Download Results

Zip everything and download to your local machine.

In [ ]:
!zip -r /content/ann_snn_results.zip results/ checkpoints/ -x "*.pdf"

from google.colab import files
files.download('/content/ann_snn_results.zip')
print("\nDownload started! Unzip in your local project to replace synthetic results.")

## 9. (Optional) Quick Validation: Does the Scaling Law Hold?

Check if accuracy really follows log₂(T) for real trained models.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

df = pd.read_csv('results/aggregated/per_run_enriched.csv')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

def log2_model(T, a, b):
    return a * np.log2(T) + b

for i, dataset in enumerate(['mnist', 'cifar10', 'cifar100']):
    ax = axes[i]
    ax.set_title(f'{dataset.upper()}: Accuracy vs T')
    
    for model in df['model'].unique():
        sub = df[(df['dataset'] == dataset) & (df['model'] == model)]
        agg = sub.groupby('T')['snn_accuracy'].mean().reset_index()
        agg = agg[agg['T'] > 1]  # skip T=1 for fitting
        
        if len(agg) >= 3:
            T_vals = agg['T'].values
            acc_vals = agg['snn_accuracy'].values * 100
            
            ax.plot(T_vals, acc_vals, 'o-', label=model, markersize=4)
            
            # Fit log2 model
            try:
                popt, _ = curve_fit(log2_model, T_vals, acc_vals)
                T_fit = np.linspace(T_vals.min(), T_vals.max(), 50)
                ax.plot(T_fit, log2_model(T_fit, *popt), '--', alpha=0.5)
            except:
                pass
    
    ax.set_xlabel('Timesteps (T)')
    ax.set_ylabel('SNN Accuracy (%)')
    ax.set_xscale('log', base=2)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/plots/validation_scaling_law.png', dpi=150)
plt.show()
print("\nIf dashed lines (log2 fit) match the dots, the scaling law HOLDS!")